# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 軌道データ

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

time_range = ['2022-09-01/22:25', '2022-09-01/23:15']

earth_radius = 6378.1  # km

psp.themis.state(trange=time_range, probe='a', no_update=True)
psp.cotrans(name_in='tha_pos_gse', name_out='tha_pos_sm', coord_in='gse', coord_out='sm')   # 'tha_pos_sm'

themis_a_pos_sm = pt.data_quants['tha_pos_sm']
themis_a_pos_sm.values = themis_a_pos_sm.values / earth_radius  # convert to RE
themis_a_pos_sm.attrs['Units'] = 'R_E'
# trangeに合わせてデータを切り出し
themis_a_pos_sm = themis_a_pos_sm.sel(time=slice(time_range[0], time_range[1]))

print(themis_a_pos_sm)

psp.erg.orb(trange=time_range, level='l2', datatype='def', no_update=True)  # 'erg_orb_l2_pos_sm'
Arase_pos_sm = pt.data_quants['erg_orb_l2_pos_sm']
Arase_pos_sm = Arase_pos_sm.sel(time=slice(time_range[0], time_range[1]))

print(Arase_pos_sm)

In [ ]:
THA_time_event  = ['2022-09-01/22:30:45', '2022-09-01/22:32:45']
time_range_T    = [THA_time_event[0].replace('/', 'T'),THA_time_event[1].replace('/', 'T')]
THA_SM_pos      = pt.data_quants['tha_pos_sm'].sel(time=slice(time_range_T[0], time_range_T[1]))

THA_rmlatmlt_R      = np.sqrt(THA_SM_pos[:, 0]**2E0 + THA_SM_pos[:, 1]**2E0 + THA_SM_pos[:, 2]**2E0)
THA_rmlatmlt_MLAT   = np.rad2deg(np.arctan2(THA_SM_pos[:, 2], np.sqrt(THA_SM_pos[:, 0]**2E0 + THA_SM_pos[:, 1]**2E0)))
THA_rmlatmlt_MLT    = np.rad2deg(np.arctan2(THA_SM_pos[:, 1], THA_SM_pos[:, 0])) / 15. + 12.

In [ ]:
print(THA_rmlatmlt_R)
print('')
print(THA_rmlatmlt_MLAT)
print('')
print(THA_rmlatmlt_MLT)

In [ ]:
Arase_time_event    = ['2022-09-01/22:30:45', '2022-09-01/22:32:45']
time_range_T        = [Arase_time_event[0].replace('/', 'T'), Arase_time_event[1].replace('/', 'T')]
Arase_rmlatmlt_pos  = pt.data_quants['erg_orb_l2_pos_rmlatmlt'].sel(time=slice(time_range_T[0], time_range_T[1]))
Arase_rmlatmlt_R    = Arase_rmlatmlt_pos[:, 0]   # Re
Arase_rmlatmlt_MLAT = Arase_rmlatmlt_pos[:, 1]   # deg
Arase_rmlatmlt_MLT  = Arase_rmlatmlt_pos[:, 2]   # hour [0,24)

In [ ]:
print(Arase_rmlatmlt_R)
print('')
print(Arase_rmlatmlt_MLAT)
print('')
print(Arase_rmlatmlt_MLT)

In [ ]:
Arase_rmlatmlt_L    = Arase_rmlatmlt_R  / (np.cos(np.deg2rad(Arase_rmlatmlt_MLAT)))**2E0
THA_rmlatmlt_L      = THA_rmlatmlt_R    / (np.cos(np.deg2rad(THA_rmlatmlt_MLAT)))**2E0

In [ ]:
print(Arase_rmlatmlt_L)
print('')
print(THA_rmlatmlt_L)

In [ ]:
Arase_rmlatmlt_L_Xsm    = Arase_rmlatmlt_L  * np.cos(np.deg2rad((Arase_rmlatmlt_MLT - 12.)*15.))
Arase_rmlatmlt_L_Ysm    = Arase_rmlatmlt_L  * np.sin(np.deg2rad((Arase_rmlatmlt_MLT - 12.)*15.))

THA_rmlatmlt_L_Xsm      = THA_rmlatmlt_L    * np.cos(np.deg2rad((THA_rmlatmlt_MLT - 12.)*15.))
THA_rmlatmlt_L_Ysm      = THA_rmlatmlt_L    * np.sin(np.deg2rad((THA_rmlatmlt_MLT - 12.)*15.))

In [ ]:
print(Arase_rmlatmlt_L_Xsm)
print('')
print(Arase_rmlatmlt_L_Ysm)
print('')
print(THA_rmlatmlt_L_Xsm)
print('')
print(THA_rmlatmlt_L_Ysm)

In [ ]:
Arase_rmlatmlt_L_Xsm_ave    = np.average(Arase_rmlatmlt_L_Xsm)
Arase_rmlatmlt_L_Ysm_ave    = np.average(Arase_rmlatmlt_L_Ysm)
print(Arase_rmlatmlt_L_Xsm_ave, Arase_rmlatmlt_L_Ysm_ave)

THA_rmlatmlt_L_Xsm_ave      = np.average(THA_rmlatmlt_L_Xsm)
THA_rmlatmlt_L_Ysm_ave      = np.average(THA_rmlatmlt_L_Ysm)
print(THA_rmlatmlt_L_Xsm_ave, THA_rmlatmlt_L_Ysm_ave)

In [ ]:
distance    = np.sqrt((Arase_rmlatmlt_L_Xsm_ave - THA_rmlatmlt_L_Xsm_ave)**2E0 + (Arase_rmlatmlt_L_Ysm_ave - THA_rmlatmlt_L_Ysm_ave)**2E0) * earth_radius
print(distance)

In [ ]:
distance / (60. * 5.)

In [ ]:
distance_Xsm    = (Arase_rmlatmlt_L_Xsm_ave - THA_rmlatmlt_L_Xsm_ave) * earth_radius
print(distance_Xsm)

In [ ]:
distance_Xsm / (60. * 5.)

In [ ]:
# 文字の大きさ
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rcParams['font.size'] = 15

fig = plt.figure(figsize=(10, 10), dpi=100)

def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

ax1 = fig.add_subplot(131)
ax1.scatter(THA_rmlatmlt_L_Xsm_ave, THA_rmlatmlt_L_Ysm_ave,  marker='o', c='yellow', s=50, edgecolors='magenta', linewidths=1)
ax1.scatter(Arase_rmlatmlt_L_Xsm_ave, Arase_rmlatmlt_L_Ysm_ave,  marker='o', c='yellow', s=50, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -9)
ax1.set_ylim(8.5, -1.5)

th = np.linspace(0, 2*np.pi, 256)
ax1.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
ax1.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')

fig.tight_layout()
plt.show()